In [ ]:
from datetime import date
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 5)

_EXTRACTED_META_COLS = ["_filter_param", "_filter_value", "_extract_datetime"]

def _add_openaire_extracted_metadata(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in _EXTRACTED_META_COLS:
        if col not in df.columns:
            df[col] = pd.NA
    return df

def _add_openaire_loaded_metadata(df: pd.DataFrame, load_datetime=None) -> pd.DataFrame:
    df = df.copy()
    if load_datetime is None:
        load_datetime = date.today()
    df["_load_datetime"] = load_datetime
    return df


In [ ]:
df = catalog.load('raw/openaire/researchproduct_dev#parquet')
df

## Paso 1: Convierto tipos y selecciono columnas con cardinalidad 1 con respecto a cada research product
+ info en https://graph.openaire.eu/docs/data-model/entities/research-product

In [ ]:
def openaire_load_researchproduct_subjects(df: pd.DataFrame)-> pd.DataFrame:
    df = _add_openaire_extracted_metadata(df)

    df_research_subjects = df.loc[:,['id','subjects', *_EXTRACTED_META_COLS]]
    df_research_subjects.dropna(inplace=True)

    df_research_subjects = df_research_subjects.explode('subjects').reset_index(drop=True)

    df_subjects = pd.json_normalize(df_research_subjects['subjects'])
    df_research_subjects = pd.concat(
        [df_research_subjects[['id', *_EXTRACTED_META_COLS]].reset_index(drop=True), df_subjects.reset_index(drop=True)],
        axis=1,
    )

    df_research_subjects = _add_openaire_loaded_metadata(df_research_subjects)

    return df_research_subjects


In [ ]:
df_research_subjects = openaire_load_researchproduct_subjects(df)

In [ ]:
df_research_subjects